In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive',force_remount=True)
print("Drive mounted successfully!")

Mounted at /content/drive
Drive mounted successfully!
Project folder found: /content/drive/MyDrive/iam_rag_project/


In [ ]:
import json
import os

def build_policy_full_text(policy):
    """
    Build rich full_text for managed policies that includes
    all statement details — actions, resources, and conditions.
    Replaces the sparse full_text from scraping which only
    contained action names.
    """
    lines = []
    lines.append(f"AWS Managed Policy: {policy['policy_name']}.")

    statements = policy.get('policy_document', {}).get('Statement', [])

    for i, statement in enumerate(statements):
        lines.append(f"Statement {i + 1}:")

        # Sid — optional human readable label
        if 'Sid' in statement:
            lines.append(f"  Purpose: {statement['Sid']}.")

        # Effect
        effect = statement.get('Effect', 'Allow')
        lines.append(f"  Effect: {effect}.")

        # Actions
        actions = statement.get('Action', [])
        if isinstance(actions, str):
            actions = [actions]
        lines.append(f"  Actions: {', '.join(actions)}.")

        # Resources
        resources = statement.get('Resource', [])
        if isinstance(resources, str):
            resources = [resources]
        lines.append(f"  Resources: {', '.join(resources)}.")

        # Conditions
        conditions = statement.get('Condition', {})
        if conditions:
            for operator, condition_block in conditions.items():
                for key, value in condition_block.items():
                    value_str = (
                        ', '.join(value)
                        if isinstance(value, list)
                        else str(value)
                    )
                    lines.append(
                        f"  Condition: {operator} "
                        f"{key} = {value_str}."
                    )

    return ' '.join(lines)

def chunk_actions(action_data):
    """
    Convert action documents into retrieval chunks.
    One chunk per action — uses full_text field directly
    as it already contains all relevant fields concatenated.
    """
    chunks = []

    for item in action_data:

        # Skip actions with no full_text or description
        if not item.get('full_text') or not item.get('description'):
            continue

        chunks.append({
            # ── Text for embedding ────────────────────────────
            # Use full_text directly — already contains all fields
            # in natural language format:
            # "IAM Action: xray:TagResource. Service: AWS X-Ray.
            #  Description: Grants permission to add tags to an
            #  X-Ray resource. Access level: Tagging.
            #  Applicable resource types: group.
            #  Supported condition keys: None."
            'text': item['full_text'],

            # ── Metadata for generation and evaluation ────────
            # Kept separate so LLM gets structured info
            # during generation step
            'metadata': {
                'type':              'iam_action',
                'doc_id':            item['doc_id'],
                'action':            item['action'],
                'action_name':       item['action_name'],
                'service':           item['service'],
                'service_prefix':    item['service_prefix'],
                'description':       item['description'],
                'access_level':      item['access_level'],
                'resource_types':    item['resource_types'],
                'condition_keys':    item['condition_keys'],
                'dependent_actions': item['dependent_actions'],
            }
        })

    return chunks

def chunk_policies(policy_data):
    """
    Convert managed policies into retrieval chunks.
    One chunk per policy — rebuilds full_text to include
    complete policy structure (actions, resources, conditions)
    rather than just action names.
    """
    chunks = []

    for policy in policy_data:

        # Skip policies with no document
        if not policy.get('policy_document'):
            continue

        chunks.append({
            # ── Text for embedding ────────────────────────────
            # Rebuilt full_text includes complete policy structure
            # including resources and conditions — not just actions
            'text': build_policy_full_text(policy),

            # ── Metadata for generation and evaluation ────────
            'metadata': {
                'type':            'iam_policy',
                'policy_name':     policy['policy_name'],
                'url':             policy.get('url', ''),
                'actions_used':    list(set(policy.get('actions_used', []))),
                # Keep full policy document for LLM generation context
                # LLM reads this JSON to understand correct policy structure
                'policy_document': policy['policy_document'],
            }
        })

    return chunks

In [ ]:
def run_chunking_pipeline():
    """
    Main chunking pipeline.
    Reads action_documents.json and managed_policies.json,
    produces action_chunks.json and policy_chunks.json.
    """

    base_dir   = "/content/drive/MyDrive/iam_rag_project/data/processed"
    output_dir = os.path.join(base_dir, "chunking")
    os.makedirs(output_dir, exist_ok=True)

    actions_path  = os.path.join(base_dir, "action_documents.json")
    policies_path = os.path.join(base_dir, "managed_policies.json")

    # ── Load data ─────────────────────────────────────────────
    print("Loading data...")
    with open(actions_path, 'r') as f:
        action_data = json.load(f)
    with open(policies_path, 'r') as f:
        policy_data = json.load(f)

    print(f"  Loaded {len(action_data)} action documents")
    print(f"  Loaded {len(policy_data)} managed policies")
    print()

    # ── Chunk actions ─────────────────────────────────────────
    print("Chunking actions...")
    action_chunks = chunk_actions(action_data)
    print(f"  ✓ {len(action_chunks)} action chunks created")

    # ── Chunk policies ────────────────────────────────────────
    print("Chunking policies...")
    policy_chunks = chunk_policies(policy_data)
    print(f"  ✓ {len(policy_chunks)} policy chunks created")

    # ── Save chunks ───────────────────────────────────────────
    action_output_path = os.path.join(output_dir, "action_chunks.json")
    policy_output_path = os.path.join(output_dir, "policy_chunks.json")

    with open(action_output_path, 'w') as f:
        json.dump(action_chunks, f, indent=2)
    with open(policy_output_path, 'w') as f:
        json.dump(policy_chunks, f, indent=2)

    print()
    print("=" * 50)
    print("CHUNKING COMPLETE")
    print("=" * 50)
    print(f"  Action chunks: {len(action_chunks)}")
    print(f"  Policy chunks: {len(policy_chunks)}")
    print(f"  Total chunks:  {len(action_chunks) + len(policy_chunks)}")
    print()
    print(f"  Saved to: {output_dir}")
    print()

    # ── Verify sample chunks ──────────────────────────────────
    print("Sample action chunk:")
    print(json.dumps(action_chunks[0], indent=2))
    print()
    print("Sample policy chunk:")
    # Show text only — policy_document JSON is long
    sample_policy = policy_chunks[0].copy()
    sample_policy['metadata'] = {
        k: v for k, v in sample_policy['metadata'].items()
        if k != 'policy_document'
    }
    sample_policy['metadata']['policy_document'] = '... (stored in metadata)'
    print(json.dumps(sample_policy, indent=2))

    return action_chunks, policy_chunks


# Run pipeline
action_chunks, policy_chunks = run_chunking_pipeline()

Loading data...
  Loaded 3094 action documents
  Loaded 500 managed policies

Chunking actions...
  ✓ 3094 action chunks created
Chunking policies...
  ✓ 500 policy chunks created

CHUNKING COMPLETE
  Action chunks: 3094
  Policy chunks: 500
  Total chunks:  3594

  Saved to: /content/drive/MyDrive/iam_rag_project/data/processed/chunking

Sample action chunk:
{
  "text": "IAM Action: apigateway:InvalidateCache. Service: Amazon API Gateway. Description: Grants permission to invalidate API cache upon a client request. Access level: Write. Applicable resource types: execute-api-general*. Supported condition keys: None.",
  "metadata": {
    "type": "iam_action",
    "doc_id": "apigateway_invalidatecache",
    "action": "apigateway:InvalidateCache",
    "action_name": "InvalidateCache",
    "service": "Amazon API Gateway",
    "service_prefix": "apigateway",
    "description": "Grants permission to invalidate API cache upon a client request",
    "access_level": "Write",
    "resource_type